# Проверка выбранной RAG-конфигурации на 10 новых вопросах

В `generation.ipynb` сравнивались модели и размеры контекста. Здесь проверяется один заранее выбранный вариант: BM25 + multilingual-e5-small, объединение RRF, переранжирование первых 20 кандидатов, YandexGPT 5 Lite 8B и 5 фрагментов в контексте. Используется текущий промпт проекта. Вопросы для подбора параметров хранятся отдельно.

**Запуск.** Откройте блокнот в корне репозитория или в каталоге `notebooks`. Понадобятся зависимости из `requirements.txt`, работающая Ollama и модель `hf.co/yandex/YandexGPT-5-Lite-8B-instruct-GGUF` (при первом запуске скачайте её командой `ollama pull hf.co/yandex/YandexGPT-5-Lite-8B-instruct-GGUF`). Энкодер и реранкер должны быть доступны локально или для загрузки. Для этой проверки подойдёт окружение с ресурсами для полной конфигурации. Веб-интерфейс и FastAPI запускать не требуется.

Выполните ячейки сверху вниз в свежем ядре. После прогона внесите ручные оценки в CSV и повторите последний раздел. Если по этим десяти вопросам изменить систему, для следующей независимой проверки понадобится новый набор.

In [ ]:
import hashlib
import json
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluate import contains, load_queries, recall_at_k, reciprocal_rank
from src.generate import INSTRUCTIONS, NOT_FOUND, OLLAMA_BASE_URL, ask_ollama, build_messages, source_label
from src.hybrid_retrieval import RERANKER, build_hybrid
from src.retrieval import load_chunks
from src.settings import settings

BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#9a9a9a"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c9c9c4",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e1",
    "axes.axisbelow": True, "font.size": 10,
})

OUT_DIR = ROOT / "data" / "generation"
TEST_PATH = ROOT / "data" / "test_queries.jsonl"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Модель и тестовые вопросы

Восемь вопросов имеют проверяемый ответ, два проверяют отказ. Для каждого обычного вопроса указаны эталон по смыслу и дословная выдержка из корпуса. В модель передаются только вопрос и найденные фрагменты.

In [ ]:
MODEL = "hf.co/yandex/YandexGPT-5-Lite-8B-instruct-GGUF"
K = 5
NUM_CTX = 8192
MODEL_OPTIONS = settings.ollama.model_dump(include={"temperature", "num_thread", "num_predict", "keep_alive"}, exclude_none=True)
if MODEL_OPTIONS != {"temperature": 0}:
    raise ValueError("Проверьте секцию [ollama] в config.toml и запустите свежее ядро с профилем default.")

search_options = dict(method="rrf", encoder_name="e5", alpha=0.6, rrf_k=60,
                      rerank=True, rerank_depth=20, bm25_field="text")
config = dict(model=MODEL, context_fragments=K, num_ctx=NUM_CTX, ollama_options=MODEL_OPTIONS,
              retrieval=search_options, reranker=RERANKER, expand=False)
pd.Series(config, name="значение").to_frame()

In [ ]:
queries = load_queries(TEST_PATH)
old = load_queries()
chunks = load_chunks()
assert len(queries) == len({q["query_id"] for q in queries}) == len({q["query"] for q in queries}) == 10
assert not ({q["query_id"] for q in queries} & {q["query_id"] for q in old})
assert not ({q["query"].strip().casefold() for q in queries} & {q["query"].strip().casefold() for q in old})
assert sum(q["category"] == "trap" for q in queries) == 2
for q in queries:
    if q["category"] == "trap":
        assert q["reference"] == NOT_FOUND and not q["evidence"]
    else:
        assert q["evidence"] and all(any(contains(c["text"], e) for c in chunks) for e in q["evidence"]), q["query_id"]
print(f"вопросов: {len(queries)}")
pd.Series([q["category"] for q in queries]).value_counts().rename("вопросов").to_frame()

## 2. Прогон выбранной конфигурации

Для оценки MRR берём первые 10 результатов. Для генерации используем только первые 5 из этой же выдачи. Время генерации включает обращение к Ollama и может включать загрузку модели. Каждый готовый ответ сохраняется сразу. Файл прогона зависит от вопросов, корпуса, промпта, настроек и реализации поиска. При продолжении незавершённого прогона дополнительно проверяется идентификатор установленной версии модели.

In [ ]:
# Изменившийся корпус, промпт или конфигурация получают отдельный файл ответов.
signature = hashlib.sha256(json.dumps([config, INSTRUCTIONS], sort_keys=True, ensure_ascii=False).encode())
for path in [TEST_PATH, ROOT / "data/chunks.jsonl", ROOT / "src/retrieval.py", ROOT / "src/hybrid_retrieval.py",
             ROOT / "src/generate.py", ROOT / "src/evaluate.py"]:
    signature.update(path.read_bytes())
ANSWERS_PATH = OUT_DIR / f"holdout_{signature.hexdigest()[:16]}.csv"
print("Файл ответов:", ANSWERS_PATH)

In [ ]:
COLUMNS = ["query_id", "category", "question", "reference", "evidence", "answer", "sources", "ranked_ids",
           "model", "model_digest", "hit_at_5", "rr_at_10", "exact_refusal", "valid_citation_numbers",
           "search_seconds", "generation_seconds", "accuracy", "hallucination", "note"]


def load_answers() -> pd.DataFrame:
    """Читает сохранённые ответы и ручную разметку."""
    if not ANSWERS_PATH.exists():
        return pd.DataFrame(columns=COLUMNS)
    df = pd.read_csv(ANSWERS_PATH, keep_default_na=False, encoding="utf-8-sig")
    if df["query_id"].duplicated().any() or not df["query_id"].isin([q["query_id"] for q in queries]).all():
        raise ValueError("В CSV обнаружены повторяющиеся или неизвестные ID.")
    return df[COLUMNS]

In [ ]:
def run_question(q: dict, retriever, digest: str) -> dict:
    """Находит десять фрагментов для оценки поиска и передаёт модели первые пять."""
    start = time.perf_counter()
    ranked = retriever.search(q["query"], k=10, expand=False)
    search_seconds = time.perf_counter() - start
    hits = ranked[:K]

    start = time.perf_counter()
    answer = ask_ollama(build_messages(q["query"], hits), model=MODEL, num_ctx=NUM_CTX)
    seconds = time.perf_counter() - start
    texts = [hit.chunk["text"] for hit in ranked]
    refs = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    trap = q["category"] == "trap"
    return {
        "query_id": q["query_id"], "category": q["category"], "question": q["query"],
        "reference": q["reference"], "evidence": "\n".join(q["evidence"]), "answer": answer,
        "sources": "\n\n".join(f"[{i}] {source_label(hit)}\n{hit.chunk['text']}" for i, hit in enumerate(hits, 1)),
        "ranked_ids": " ".join(hit.chunk_id for hit in ranked), "model": MODEL, "model_digest": digest,
        "hit_at_5": None if trap else recall_at_k(texts, q["evidence"], k=5),
        "rr_at_10": None if trap else reciprocal_rank(texts, q["evidence"], k=10),
        "exact_refusal": int(answer.strip() == NOT_FOUND),
        "valid_citation_numbers": int(bool(refs) and all(1 <= n <= len(hits) for n in refs)),
        "search_seconds": round(search_seconds, 3), "generation_seconds": round(seconds, 2),
        "accuracy": "", "hallucination": "", "note": "",
    }

In [ ]:
def run() -> None:
    """Получает недостающие ответы. Сохраняет результат после каждого вопроса."""
    done = load_answers()
    done_ids = set(done["query_id"])
    todo = [q for q in queries if q["query_id"] not in done_ids]
    if not todo:
        print("Все ответы уже сохранены")
        return

    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    response.raise_for_status()
    models = {m["name"]: m["digest"] for m in response.json()["models"]}
    digest = models.get(MODEL) or models.get(MODEL + ":latest")
    if not digest:
        raise RuntimeError(f"Модель отсутствует в Ollama. Выполните ollama pull {MODEL}")
    if not done["model_digest"].eq(digest).all():
        raise ValueError("Версия модели изменилась. Сохраните предыдущий CSV отдельно перед новым прогоном.")

    retriever = build_hybrid(**search_options)
    rows = done.to_dict("records")
    for i, q in enumerate(todo, start=1):
        try:
            row = run_question(q, retriever, digest)
        except (RuntimeError, requests.RequestException) as error:
            print(f"{q['query_id']} пропущен, {error}")
            continue
        rows.append(row)
        tmp = ANSWERS_PATH.with_suffix(".tmp")
        pd.DataFrame(rows, columns=COLUMNS).to_csv(tmp, index=False, encoding="utf-8-sig")
        tmp.replace(ANSWERS_PATH)
        print(f"{i}/{len(todo)} {q['query_id']} {row['generation_seconds']:.1f} с")

In [ ]:
run()

## 3. Качество поиска и формат ответа

Recall@5 и MRR@10 считаются по восьми вопросам с ответом, распознавание ловушек — по двум. Для автоматической метрики отказа используется точное совпадение с фразой из промпта. Проверка номеров ссылок показывает только соблюдение формата. Смысл ответа и обоснованность ссылок проверяются по сохранённому контексту.

In [ ]:
def load_typed() -> pd.DataFrame:
    """Приводит оценки и метрики к числам. Пустые поля разметки остаются NaN."""
    df = load_answers()
    for column in ["hit_at_5", "rr_at_10", "exact_refusal", "valid_citation_numbers", "generation_seconds",
                   "accuracy", "hallucination"]:
        df[column] = pd.to_numeric(df[column].replace("", float("nan")), errors="raise")
    for column, allowed in {"accuracy": [0, 1, 2], "hallucination": [0, 1]}.items():
        if not df[column].dropna().isin(allowed).all():
            raise ValueError(f"Допустимые значения {column}: {allowed}")
    return df


def format_metrics(df: pd.DataFrame) -> pd.Series:
    """Считает качество поиска на обычных вопросах и отказы на ловушках."""
    regular = df[df["category"] != "trap"]
    traps = df[df["category"] == "trap"]
    return pd.Series({
        "Recall@5": regular["hit_at_5"].mean(),
        "MRR@10": regular["rr_at_10"].mean(),
        "точные отказы на ловушках (из 2)": traps["exact_refusal"].sum(),
        "точные отказы на обычных вопросах (из 8)": regular["exact_refusal"].sum(),
        "ответы с допустимыми ссылками (из 8)": regular["valid_citation_numbers"].sum(),
        "медианное время генерации, с": df["generation_seconds"].median(),
    })


answers = load_typed()
print(f"получено ответов: {len(answers)} из {len(queries)}")
if len(answers) == len(queries):
    display(format_metrics(answers).round(3).to_frame("значение"))
else:
    print("Для итоговых метрик повторите ячейку run() и завершите прогон.")

## 4. Разметка ответов

Откройте CSV по пути из раздела 2. Сопоставьте `answer` с эталоном `reference` и фрагментами `sources`. Заполните `accuracy` (2 — полный верный ответ, 1 — частичный, 0 — неверный или ложный отказ) и `hallucination` (1 — есть фактическое утверждение без опоры на контекст, 0 — таких утверждений нет). На ловушке обоснованный отказ получает 2 балла; оцените смысл даже при другой формулировке. В `note` отметьте причину ошибки или некорректную ссылку. Сохраните CSV и повторите ячейки разделов 4 и 5. Один вопрос составляет 10 процентных пунктов итоговой точности.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(answers[["query_id", "category", "question", "reference", "answer", "hit_at_5"]])

In [ ]:
def manual_metrics(df: pd.DataFrame) -> pd.Series:
    """Считает точность и долю галлюцинаций по ручной разметке."""
    return pd.Series({
        "точность, все (0–2)": df["accuracy"].mean(),
        "верных ответов, все": (df["accuracy"] == 2).mean(),
        "галлюцинации": df["hallucination"].mean(),
        "ловушки распознаны": (df.loc[df["category"] == "trap", "accuracy"] == 2).mean(),
    })


answers = load_typed()
marked = answers.dropna(subset=["accuracy", "hallucination"])
print(f"размечено: {len(marked)} из {len(queries)}")
if len(marked) == len(queries):
    display(manual_metrics(marked).round(3).to_frame("значение"))
else:
    print("Для итоговой оценки заполните accuracy и hallucination в CSV.")

In [ ]:
def plot_accuracy(df: pd.DataFrame) -> None:
    """Показывает, сколько ответов получили каждый балл ручной оценки."""
    counts = df["accuracy"].value_counts().reindex([0, 1, 2], fill_value=0)
    fig, ax = plt.subplots(figsize=(7, 2.6))
    bars = ax.barh(["Неверный ответ", "Частичный ответ", "Верный ответ"], counts, color=[GREY, ORANGE, BLUE])
    ax.bar_label(bars, padding=3)
    ax.set_xlim(0, len(df) + 1)
    ax.set_xlabel("число ответов")
    ax.set_title("Оценка ответов на тестовых вопросах", loc="left")
    plt.tight_layout()


if len(marked) == len(queries):
    plot_accuracy(marked)

## 5. Разбор ошибок

Для каждого ошибочного или частичного ответа сопоставьте эталон с текстом модели и источниками из CSV. Признак `hit_at_5` показывает, попал ли в контекст фрагмент с размеченной фразой. Для ловушек этот показатель не вычисляется.

In [ ]:
errors = marked[(marked["accuracy"] < 2) | (marked["hallucination"] == 1)]
print(f"ошибочных ответов: {len(errors)} из {len(marked)} размеченных")
with pd.option_context("display.max_colwidth", None):
    display(errors[["query_id", "question", "reference", "answer", "hit_at_5", "accuracy", "hallucination", "note"]])